In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config


Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
# — Silver | Join all sources into one film master table
from pyspark.sql import functions as F

# Load all tables
df_trends  = spark.table(f"{SILVER_PATH}.trends_silver")
df_post    = spark.table(f"{SILVER_PATH}.trends_postrelease_silver")
df_youtube = spark.table(f"{SILVER_PATH}.youtube_silver")
df_imdb    = spark.table(f"{BRONZE_PATH}.imdb_ratings_raw")

# — Fix title mismatches in YouTube table
title_fixes = [
    ("Top Gun Maverick",              "Top Gun: Maverick"),
    ("Mad Max Fury Road",             "Mad Max: Fury Road"),
    ("Indiana Jones Dial of Destiny", "Indiana Jones and the Dial of Destiny"),
]

for old, new in title_fixes:
    df_youtube = df_youtube.withColumn("film",
        F.when(F.col("film") == old, new).otherwise(F.col("film")))

# — Join all sources on film title
df_master = (df_trends
    .join(df_post.select(
            "film", "post_peak_score", "post_avg_score",
            "days_above_50_post", "avg_first30_post"),
          on="film", how="left")
    .join(df_youtube.drop("release_date", "genre", "era"),
          on="film", how="left")
    .join(df_imdb.select(
            "film", "imdb_rating", "vote_count",
            "runtime_minutes", "genres"),
          on="film", how="left")
)

# Normalize IMDb rating to 0-100 scale
df_master = df_master.withColumn(
    "reaction_score",
    (F.col("imdb_rating") / 10 * 100).cast("double")
)

# Calculate attention decay rate
# How much interest dropped from pre-release peak to post-release first 30 days
df_master = df_master.withColumn(
    "attention_decay_rate",
    F.when(
        F.col("hype_peak_score") > 0,
        ((F.col("hype_peak_score") - F.col("avg_first30_post")) /
          F.col("hype_peak_score") * 100).cast("double")
    ).otherwise(None)
)

# Final column selection
df_master = df_master.select(
    "film",
    "release_date",
    "genre",
    "era",
    "runtime_minutes",
    "genres",
    # Pre-release hype signals
    "hype_peak_score",
    "hype_avg_score",
    "days_above_50",
    "view_count",
    "view_count_norm",
    "engagement_rate",
    "days_trailer_to_release",
    # Post-release signals
    "post_peak_score",
    "post_avg_score",
    "avg_first30_post",
    "attention_decay_rate",
    # Reaction signals
    "imdb_rating",
    "reaction_score",
    "vote_count"
).orderBy("release_date")

print(f"✅ Film master ready — {df_master.count()} films")
display(df_master)

✅ Film master ready — 50 films


film,release_date,genre,era,runtime_minutes,genres,hype_peak_score,hype_avg_score,days_above_50,view_count,view_count_norm,engagement_rate,days_trailer_to_release,post_peak_score,post_avg_score,avg_first30_post,attention_decay_rate,imdb_rating,reaction_score,vote_count
Batman Begins,2005-06-15,Action,Pre Short-Form (2005-2014),140,"Action,Crime,Drama",100,11.846153846153847,2,6244470,7.166475647717892,0.839831082541833,-3102,100,15.912087912087912,35.903225806451616,64.09677419354838,8.2,82.0,1703636
Brokeback Mountain,2005-12-09,Drama,Pre Short-Form (2005-2014),134,"Drama,Romance",100,12.87912087912088,1,814753,0.9350525398320587,0.5119342917424055,-2195,100,45.142857142857146,57.83870967741935,42.16129032258065,7.7,77.0,414988
The Departed,2006-10-06,Drama,Pre Short-Form (2005-2014),151,"Crime,Drama,Thriller",100,6.450549450549451,1,5249806,6.024947970643349,0.5467630613397905,-2779,100,19.483516483516482,40.354838709677416,59.64516129032258,8.5,85.0,1545039
Casino Royale,2006-11-17,Action,Pre Short-Form (2005-2014),144,"Action,Adventure,Thriller",100,9.516483516483516,1,5596494,6.422824989726797,0.5254539717187224,-2083,100,14.791208791208792,30.612903225806452,69.38709677419355,8.0,80.0,740706
Ratatouille,2007-06-29,Comedy,Pre Short-Form (2005-2014),111,"Adventure,Animation,Comedy",100,8.758241758241759,1,7314038,8.393967016173233,0.25723136795296936,-3785,100,17.9010989010989,37.0,63.0,8.1,81.0,935005
No Country for Old Men,2007-11-09,Thriller,Pre Short-Form (2005-2014),122,"Crime,Drama,Thriller",100,5.637362637362638,1,5786136,6.640467924160707,0.6196190341879279,-2374,100,37.67032967032967,48.483870967741936,51.516129032258064,8.2,82.0,1179869
There Will Be Blood,2007-12-26,Drama,Pre Short-Form (2005-2014),158,Drama,100,9.736263736263735,4,3958323,4.542775509418994,0.6655091057500866,-2327,100,33.62637362637363,41.16129032258065,58.83870967741935,8.2,82.0,703599
The Dark Knight,2008-07-18,Action,Pre Short-Form (2005-2014),152,"Action,Crime,Drama",100,9.010989010989011,1,21476936,24.64803879778359,1.034863632317012,-1964,100,12.813186813186814,27.870967741935484,72.12903225806451,9.1,90.99999999999999,3145602
The Hangover,2009-06-05,Comedy,Pre Short-Form (2005-2014),100,Comedy,100,7.0989010989010985,1,5851864,6.715900764957956,0.5154938665696946,-1629,100,25.36263736263736,47.935483870967744,52.064516129032256,7.7,77.0,923449
Paranormal Activity,2009-10-16,Horror,Pre Short-Form (2005-2014),86,"Horror,Mystery",100,11.021978021978022,10,374154,0.429398416438263,0.5527135885223731,-994,100,17.439560439560438,39.38709677419355,60.61290322580645,6.3,63.0,271193


In [0]:
(df_master
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_PATH}.film_master")
)

print(f"✅ Saved to {SILVER_PATH}.film_master")
print(f"   Rows written: {df_master.count()}")

display(spark.table(f"{SILVER_PATH}.film_master").limit(5))

✅ Saved to bootcamp_students.tiffani_ceresrain_silver.film_master
   Rows written: 50


film,release_date,genre,era,runtime_minutes,genres,hype_peak_score,hype_avg_score,days_above_50,view_count,view_count_norm,engagement_rate,days_trailer_to_release,post_peak_score,post_avg_score,avg_first30_post,attention_decay_rate,imdb_rating,reaction_score,vote_count
Batman Begins,2005-06-15,Action,Pre Short-Form (2005-2014),140,"Action,Crime,Drama",100,11.846153846153847,2,6244470,7.166475647717892,0.839831082541833,-3102,100,15.912087912087912,35.903225806451616,64.09677419354838,8.2,82.0,1703636
Brokeback Mountain,2005-12-09,Drama,Pre Short-Form (2005-2014),134,"Drama,Romance",100,12.87912087912088,1,814753,0.9350525398320587,0.5119342917424055,-2195,100,45.142857142857146,57.83870967741935,42.16129032258065,7.7,77.0,414988
The Departed,2006-10-06,Drama,Pre Short-Form (2005-2014),151,"Crime,Drama,Thriller",100,6.450549450549451,1,5249806,6.024947970643349,0.5467630613397905,-2779,100,19.483516483516482,40.354838709677416,59.64516129032258,8.5,85.0,1545039
Casino Royale,2006-11-17,Action,Pre Short-Form (2005-2014),144,"Action,Adventure,Thriller",100,9.516483516483516,1,5596494,6.422824989726797,0.5254539717187224,-2083,100,14.791208791208792,30.612903225806452,69.38709677419355,8.0,80.0,740706
Ratatouille,2007-06-29,Comedy,Pre Short-Form (2005-2014),111,"Adventure,Animation,Comedy",100,8.758241758241759,1,7314038,8.393967016173233,0.25723136795296936,-3785,100,17.9010989010989,37.0,63.0,8.1,81.0,935005
